# Processing BIOMASS Level-1a data

This tutorial reads a local BIOMASS Level-1a Standard SCS product and builds a lazy polarimetric processing pipeline:

1. reconstruct the scattering matrix with `open_biomass_l1a`;
2. convert it to a `T3` coherency matrix;
3. multilook to approximately square pixels in SAR geometry;
4. apply the polarimetric Refined Lee speckle filter; and
5. compute and display the H/A/Alpha decomposition.

In [ ]:
from pathlib import Path

import dask
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.decompositions import h_a_alpha
from polsarpro.io import open_biomass_l1a
from polsarpro.speckle_filters import refined_lee
from polsarpro.util import S_to_T3, multilook, pauli_rgb

## Open a BIOMASS product

Point `product_path` to an unzipped `S[123]_SCS__1S` product directory. The reader finds the amplitude and phase rasters under `measurement/`, reconstructs the four complex channels, and keeps them lazy using Dask.

In [ ]:
product_path = Path(
    "/data/psp/test_files/"
    "BIO_S2_SCS__1S_20251216T034800_20251216T034815_"
    "T_G01_M01_C02_T017_F289_01_DJQGAN"
)

S = open_biomass_l1a(product_path, chunks={"y": 900, "x": 600})
S

## Convert to a coherency matrix

In [ ]:
T3 = S_to_T3(S)
T3

## Apply azimuth multilooking 

In [ ]:
T3_multilooked = multilook(
    T3, dim_az=3, dim_rg=1
)
T3_multilooked

## Apply the Refined Lee filter

In [ ]:
T3_filtered = refined_lee(
    T3_multilooked,
    window_size=7,
    num_looks=3,
)

## Apply H/A/Alpha

The Refined Lee result is already spatially filtered, so the decomposition uses a 1 × 1 boxcar to avoid adding another averaging step.

In [ ]:
haa = h_a_alpha(
    T3_filtered,
    boxcar_size=[1, 1],
    flags=("entropy", "anisotropy", "alpha"),
)

## Make RGB visuals for intermediate products

In [ ]:
rgb_multilooked = pauli_rgb(T3_multilooked)
rgb_filtered = pauli_rgb(T3_filtered)

## Compute results

The processing graph is evaluated only here. Computing the two Pauli images and the decomposition together allows Dask to share upstream work.

In [ ]:
with ProgressBar():
    rgb_multilooked, rgb_filtered, haa = dask.compute(
        rgb_multilooked, rgb_filtered, haa
    )

## Display the results

First compare Pauli RGB images before and after speckle filtering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
rgb_multilooked.plot.imshow(ax=axes[0], rgb="band", add_labels=False)
rgb_filtered.plot.imshow(ax=axes[1], rgb="band", add_labels=False)
axes[0].set_title("Multilooked T3")
axes[1].set_title("Refined Lee filtered T3")
plt.show()

Finally display entropy, anisotropy, and mean alpha angle.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
haa.entropy.plot.imshow(ax=axes[0], vmin=0, vmax=1, cmap="viridis")
haa.anisotropy.plot.imshow(ax=axes[1], vmin=0, vmax=1, cmap="viridis")
haa.alpha.plot.imshow(ax=axes[2], vmin=0, vmax=90, cmap="turbo")
axes[0].set_title("Entropy (H)")
axes[1].set_title("Anisotropy (A)")
axes[2].set_title("Mean alpha angle (°)")
plt.show()